# Notebook 3: System Performance Evaluation & Latency BenchmarksThis notebook measures retrieval accuracy (Precision@1, Precision@3)and query execution latency across operational test suites.---

## 1. Setup & Evaluation Suite

In [1]:
import osimport jsonimport timeimport reimport numpy as npimport pandas as pdfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similarityKB_PATH = os.path.join('..', 'data', 'insurance_knowledge_base.json')if not os.path.exists(KB_PATH):    KB_PATH = 'insurance_knowledge_base.json'with open(KB_PATH, 'r', encoding='utf-8') as f:    kb = json.load(f)def normalize_arabic(text):    if not text: return ''    text = re.sub(r'[\u064B-\u0652\u0670]', '', text)    text = re.sub(r'[أإآ]', 'ا', text)    text = text.replace('ة', 'ه').replace('ى', 'ي').lower()    return textdocuments, chunk_index = [], []STOP_WORDS = {'في','من','على','إلى','عن','مع','هذا','هذه','التي','الذي','هو','هي','أن','كان','كل','لم','لن','يتم','يجب','لابد','و','أو','لا','ما','فى','بعد','قبل'}for ck, cd in kb.items():    for cat, pol in cd.get('policies', {}).items():        chunk = ' '.join([cd.get('company_name', ''), cat, pol.get('details', ''), pol.get('notes', '')])        documents.append(normalize_arabic(chunk))        chunk_index.append((ck, cat))vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words=list(STOP_WORDS))tfidf_matrix = vectorizer.fit_transform(documents)print(f'TF-IDF Matrix built: {tfidf_matrix.shape}')

TF-IDF Matrix built: (772, 5000)


## 2. Accuracy & Latency Benchmarks

In [1]:
test_suite = [    ('محظورات يونايتد', 'يونايتد', 'المحظورات'),    ('أقصى مدة صرف ويبكو', 'ويبكو', 'أقصى مدة للصرف'),    ('تواصل موافقات دريم مشرق', 'دريم مشرق', 'التواصل للموافقات'),    ('ختم جلوبميد', 'جلوبميد', 'الختم'),    ('تحمل يونايتد', 'يونايتد', 'التحمل'),    ('نماذج صرف المشرق', 'المشرق', 'نماذج الصرف'),    ('صورة كارنيه يونيكير', 'يونيكير', 'صورة الكارنية'),    ('صلاحية نموذج منصور', 'منصور', 'صلاحية النموذج'),    ('محظورات اليكو', 'أليكو', 'المحظورات'),    ('تشخيص يونايتد', 'يونايتد', 'التشخيص'),]correct_p1, correct_p3 = 0, 0latencies = []for q, exp_comp, exp_cat in test_suite:    t0 = time.time()    q_v = vectorizer.transform([normalize_arabic(q)])    sims = cosine_similarity(q_v, tfidf_matrix).flatten()    top3 = sims.argsort()[-3:][::-1]    dt = (time.time() - t0) * 1000.0  # ms    latencies.append(dt)    p1_match = False    p3_match = False    for i, idx in enumerate(top3):        ck, cat = chunk_index[idx]        comp_name = kb[ck].get('company_name', '')        if exp_comp.lower() in comp_name.lower() and exp_cat.lower() in cat.lower():            if i == 0: p1_match = True            p3_match = True    if p1_match: correct_p1 += 1    if p3_match: correct_p3 += 1print(f'Precision @ 1: {correct_p1 / len(test_suite):.1%}')print(f'Precision @ 3: {correct_p3 / len(test_suite):.1%}')print(f'Average Latency: {np.mean(latencies):.2f} ms')print(f'Max Latency: {np.max(latencies):.2f} ms')

Precision @ 1: 10.0%
Precision @ 3: 20.0%
Average Latency: 0.50 ms
Max Latency: 1.00 ms


## 3. Metric Summary

In [1]:
summary_df = pd.DataFrame([    {'Metric': 'Total Insurance Entities', 'Value': str(len(kb))},    {'Metric': 'Total Document Chunks', 'Value': str(len(documents))},    {'Metric': 'Precision @ 1', 'Value': f'{correct_p1 / len(test_suite):.1%'}},    {'Metric': 'Precision @ 3', 'Value': f'{correct_p3 / len(test_suite):.1%'}},    {'Metric': 'Average Query Latency', 'Value': f'{np.mean(latencies):.2f} ms'},])summary_df

Cell Note: closing parenthesis '}' does not match opening parenthesis '[' on line 1 (<string>, line 4)
